# 3. Search check: the search links

The search check was done in a browser: automated Google searches are blocked with a CAPTCHA. This notebook builds the two links for each of the 31 stories in `data/search_check/search_stories.csv`:
- Google News search;
- Google Search's News tab.

Both are limited to the story's dates. Open each link, then record the top 10 results (rank, outlet, headline, time) in the format of `data/search_check/search_results.csv`. Mark results about a different story as `on_story = False`.

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'collection' else Path.cwd()   # repository root
RAW, CACHE, OUT = ROOT / 'data' / 'raw', ROOT / 'cache', ROOT / 'outputs'
CACHE.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)
from urllib.parse import quote_plus
import pandas as pd

stories = pd.read_csv(ROOT / 'data' / 'search_check' / 'search_stories.csv')
def gnews(r):
    lang = 'hl=fr-CA&gl=CA&ceid=CA:fr' if r.edition == 'fr-CA' else 'hl=en-CA&gl=CA&ceid=CA:en'
    return f'https://news.google.com/search?q={quote_plus(f"{r.query} after:{r.date_min} before:{r.date_max}")}&{lang}'
def mdy(d):
    t = pd.Timestamp(d)
    return f'{t.month}/{t.day}/{t.year}'
def gsearch(r):
    d0, d1 = mdy(r.date_min), mdy(r.date_max)
    lang = 'hl=fr&gl=ca' if r.edition == 'fr-CA' else 'hl=en&gl=ca'
    return f'https://www.google.com/search?q={quote_plus(r.query)}&tbm=nws&tbs=cdr:1,cd_min:{d0},cd_max:{d1}&{lang}'

links = stories.assign(google_news=[gnews(r) for r in stories.itertuples()],
                       google_search=[gsearch(r) for r in stories.itertuples()])
links[['story_id', 'edition', 'query', 'google_news', 'google_search']].to_csv(OUT / 'search_links.csv', index=False)
links[['story_id', 'query', 'google_news']].head()

## Step 2. Data report: missing values, duplicates, example rows

Reads the finished tables in `data/search_check/`.
Saves `outputs/search_missingness.csv` and prints the numbers used in Section 2 of the report.

In [1]:
from pathlib import Path
import pandas as pd

# Setting root
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'search_check').is_dir())
SC = ROOT / 'data' / 'search_check'
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)

tables = {'search_stories': pd.read_csv(SC / 'search_stories.csv'),
          'search_results': pd.read_csv(SC / 'search_results.csv')}
st, res = tables['search_stories'], tables['search_results']

# missing values: every column with any, saved for the report
miss = pd.concat([pd.DataFrame({'table': n, 'column': d.columns, 'n_missing': d.isna().sum().values,
                                'pct_missing': (d.isna().mean() * 100).round(1).values}) for n, d in tables.items()])
miss = miss[miss.n_missing > 0]
miss.to_csv(OUT / 'search_missingness.csv', index=False)

print('Stories:', len(st), st.edition.value_counts().to_dict(), '| found via:', st.found_via.value_counts().to_dict())
print('Results:', len(res), res.engine.value_counts().to_dict(),
      '| results per query (median):', res.groupby(['story_id', 'engine']).size().median())
print('Missing publish time by engine:', res[res.published.isna()].engine.value_counts().to_dict())
print('Off-story results (label noise, dropped):', int((~res.on_story).sum()), f'({(~res.on_story).mean():.1%})')
print('Results by size group:', res.size_group.value_counts().to_dict())
print('Same result twice in one query:', res.duplicated(['story_id', 'engine', 'source', 'headline']).sum())

# example rows (section 2.3)
display(res.head(3))
miss

Stories: 31 {'en-CA': 25, 'fr-CA': 6} | found via: {'local': 18, 'archive': 12, 'today': 1}
Results: 487 {'gsearch': 246, 'gnews': 241} | results per query (median): 10.0
Missing publish time by engine: {'gsearch': 246}
Off-story results (label noise, dropped): 75 (15.4%)
Results by size group: {'major': 282, 'small': 124, 'other': 81}
Same result twice in one query: 2


,story_id,engine,rank,source,headline,published,group,on_story,size_group
0,b3c9f581,gnews,1,Northumberland News,"Cobourg names its first woman deputy chief, br...",2026-10-08T15:31,major,True,major
1,b3c9f581,gnews,2,GRANT HAVEN MEDIA,Cobourg Police Service Board names new deputy ...,2026-10-09T12:52,local_en,True,small
2,b3c9f581,gnews,3,PTBO Today,Former Peterborough officer Tina Maier named C...,2026-10-08T11:39,local_en,True,small


,table,column,n_missing,pct_missing
9,search_stories,major_time,12,38.7
5,search_results,published,246,50.5
